# Customer Segmentation AnalysisRFM analysis + K-means clustering for customer segmentation.

In [ ]:
# Import librariesimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.preprocessing import StandardScalerfrom sklearn.cluster import KMeansplt.style.use("seaborn-v0_8-whitegrid")

## 1. Load Data

In [ ]:
df = pd.read_csv('data/customer_transactions.csv')df['purchase_date'] = pd.to_datetime(df['purchase_date'])print('Shape:', df.shape)df.head()

## 2. RFM Analysis

In [ ]:
analysis_date = df['purchase_date'].max() + pd.Timedelta(days=1)rfm = df.groupby('customer_id').agg({    'purchase_date': lambda x: (analysis_date - x.max()).days,    'transaction_id': 'count',    'amount': 'sum'}).reset_index()rfm.columns = ['customer_id', 'recency', 'frequency', 'monetary']print(rfm.head())

## 3. RFM Scoring & Segmentation

In [ ]:
rfm['R_score'] = pd.qcut(rfm['recency'], q=5, labels=[5,4,3,2,1], duplicates='drop')rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1,2,3,4,5])rfm['RFM_score'] = rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)def segment(s):    if s >= 13: return 'Champions'    elif s >= 10: return 'Loyal'    elif s >= 7: return 'Potential'    elif s >= 5: return 'At Risk'    else: return 'Lost'rfm['segment'] = rfm['RFM_score'].apply(segment)print(rfm['segment'].value_counts())

## 4. K-means Clustering

In [ ]:
X = rfm[['recency', 'frequency', 'monetary']]scaler = StandardScaler()X_scaled = scaler.fit_transform(X)kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)rfm['cluster'] = kmeans.fit_predict(X_scaled)print(rfm.groupby('cluster')[['recency', 'frequency', 'monetary']].mean())

## 5. Visualization

In [ ]:
rfm['segment'].value_counts().plot(kind='bar', figsize=(10,5), color=['#2ecc71','#3498db','#f39c12','#e74c3c','#9b59b6'])plt.title('Customer Segments Distribution')plt.ylabel('Count')plt.tight_layout()plt.show()